# D78 架构消融: 四编码器 BC 闭环对比(val)

ckpt 全部现成(D59 时代训好), 只做闭环评估, 每个几分钟。开环结论(M11: 四家 tie, macro 0.85–0.86)已可引用; 本表把「架构次要、GNN 选择不影响结论」升级为闭环证据。主方法不做 ×4(机时不允许, 论文注明 scope)。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
print('OK')


## 评估(4 编码器 × val 11 天)

In [ ]:
import json, torch
from derbysim.learn.features import Featurizer
from derbysim.learn.panel import PanelAdapter
from derbysim.learn.panel_models import PanelOptionPolicy
from derbysim.learn.ppo import TorchPanelDriver
from derbysim.learn.rollout import as_factory, evaluate_policy
from derbysim.learn.scenarios import build_day_trains, day_horizon, load_assets, operating_days
from derbysim.sim.reward import RewardedDynamicEnv
net,params,dwell,ptc=load_assets()
ad=PanelAdapter(net,Featurizer(net,ptc))   # BC = fv1 口径
envf=lambda: RewardedDynamicEnv(net,params,ptc,dwell,proactive=True,reserve_window='wide',control_classes={'1','2'})
eps=[(str(w['start'])[:16],(lambda w=w: build_day_trains(w)),day_horizon(w)) for w in operating_days('val')]
out={}
for enc in ['gnn','mlp','deepsets','settransformer']:
    ck=torch.load(f'artifacts/dataset/bc_fullpanel_{enc}.pt',map_location='cpu',weights_only=False)
    pol=PanelOptionPolicy(ck.get('encoder',enc),*ck['dims'],hidden=64)
    pol.load_state_dict(ck['state_dict']); pol.eval()
    ev=evaluate_policy(envf,ad,as_factory(TorchPanelDriver(pol,torch.device('cpu'),greedy=True)),eps)
    ev.pop('eval_episodes',None); out[enc]=ev
    print(f"{enc:<16} 完成 {ev['eval_completion_rate']:.4f} | 延误 {ev['eval_weighted_secondary_delay_s']:>9.0f} | reward {ev['eval_episode_reward']:>8.1f} | WAIT {ev['eval_frac_wait']:.0%}")
json.dump(out,open('artifacts/reports/bc_encoders_closedloop.json','w'))
print('saved -> artifacts/reports/bc_encoders_closedloop.json')
